# chameleon: style cartridges for Qwen-Image 2.1

Upload a few images whose style you like. Chameleon writes that style out as a short description (a *cartridge*)
and uses it to generate new images, or to restyle your photos, with Qwen-Image 2.1 Turbo.

Needs a GPU with at least 22 GB (L4 or A100: *Runtime → Change runtime type*). The first run downloads about 40 GB.
Research use only: the head was trained on MegaStyle-1.4M, which has a research licence.

In [ ]:
#@title 1 · setup
import os, subprocess, sys
subprocess.run("pip uninstall -y -q torchao", shell=True)
subprocess.run('pip install -q "git+https://github.com/huggingface/diffusers@85c9fa4caee33d1c5b473395784471d2d7957c1b" '
               '"transformers==5.19.0" accelerate safetensors sentencepiece', shell=True, check=True)
if not os.path.isdir("/content/chameleon"):
    subprocess.run(["git", "clone", "-q", "https://github.com/kelechi-c/chameleon", "/content/chameleon"], check=True)
os.chdir("/content/chameleon")
sys.path.insert(0, "/content/chameleon/src")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from huggingface_hub import hf_hub_download
HEAD = hf_hub_download("tensorkelechi/chameleon-ar-head", "head_pre.pt")
print("ready")

In [ ]:
#@title 2 · upload style images (then, optionally, photos to restyle)
ONE_STYLE = False  #@param {type:"boolean"}
RESTYLE_PHOTOS = False  #@param {type:"boolean"}
import io, pathlib as pl
from PIL import Image
from google.colab import files
from chameleon.serve import square
load = lambda up: {pl.Path(n).stem: Image.open(io.BytesIO(b)).convert("RGB") for n, b in up.items()}
styles = load(files.upload())
REFS = {"style": list(styles.values())} if ONE_STYLE else {n: [im] for n, im in styles.items()}
PHOTOS = [square(im, 1024) for im in load(files.upload()).values()] if RESTYLE_PHOTOS else []
print(len(REFS), "styles,", len(PHOTOS), "photos")

In [ ]:
#@title 3 · settings
PROMPTS = "a red fox in a snowy forest | a lighthouse on a rocky coast"  #@param {type:"string"}
SEED = 7  #@param {type:"integer"}
TEXTURE_BOOST = True  #@param {type:"boolean"}
EDIT_STRENGTH = 1.0  #@param {type:"slider", min:0.25, max:2.0, step:0.25}
MIX_FIRST_TWO = False  #@param {type:"boolean"}
PROMPTS = [p.strip() for p in PROMPTS.split("|") if p.strip()]

`TEXTURE_BOOST` also shows the model your style image in a few middle layers, which adds surface texture without
copying its subject. `EDIT_STRENGTH` above 1 pushes the style harder on photos. `MIX_FIRST_TWO` blends the first two
styles 50/50.

In [ ]:
#@title 4 · load the models (once)
from chameleon import serve
pipe = serve.Pipeline(HEAD, None)

In [ ]:
#@title 5 · run
from IPython.display import display
OUT = pl.Path("/content/chameleon_out"); OUT.mkdir(exist_ok=True)
carts = {n: pipe.make(ims, [n] * len(ims)) for n, ims in REFS.items()}
if MIX_FIRST_TWO and len(carts) >= 2:
    a, b = list(carts)[:2]
    carts[f"{a}+{b}"] = pipe.mix([carts[a], carts[b]], [0.5, 0.5])
for n, c in carts.items():
    ref = REFS[n][0] if n in REFS else None
    said, out = pipe.run(c, PROMPTS, PHOTOS, ref, SEED, "text", TEXTURE_BOOST, EDIT_STRENGTH)
    c.describe = said
    c.save(OUT / f"{n}.cart")
    print(f"\n[{n}] {said}")
    row = Image.new("RGB", (256 * len(out), 256), "white")
    for j, (label, im) in enumerate(out):
        im.save(OUT / f"{n}_{j:02d}.jpg", quality=95)
        row.paste(square(im, 256), (256 * j, 0))
    print(" | ".join(label for label, _ in out))
    display(row)

In [ ]:
#@title 6 · download (images + .cart files)
import shutil
shutil.make_archive("/content/chameleon_out", "zip", OUT)
files.download("/content/chameleon_out.zip")

A `.cart` file is a style you can keep and share: `serve.cg.Cartridge.load("x.cart")` brings it back.